# Faster R-CNN (MobileNetV3-Large-FPN) on Pascal VOC 2012 — experiments

Second detector, compared against YOLO26. Labels are read **directly from the VOC XML files**, with the **same split as the YOLO notebook**
(trainval.txt sorted, shuffled with seed 42, 80/20) and the same Kaggle test set. Difficult objects are kept, as in YOLO.
Training loop, augmentation, scheduler and evaluation are written out here; torchvision only provides the model.

**mAP is computed with torchmetrics (COCO-style, pycocotools backend), not with Ultralytics.**
The two implementations differ slightly, so YOLO and Faster R-CNN numbers are compared with that caveat.

## Install
PyTorch and torchvision are preinstalled on Kaggle. torchmetrics needs pycocotools for detection mAP.

In [ ]:
!pip install -q -U wandb torchmetrics pycocotools

## Imports

In [ ]:
import math
import random
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import torch
import wandb
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection import MeanAveragePrecision
from torchvision.io import ImageReadMode, read_image
from torchvision.models.detection import (FasterRCNN_MobileNet_V3_Large_FPN_Weights,
                                          fasterrcnn_mobilenet_v3_large_fpn)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import ColorJitter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## W&B login

In [ ]:
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("wandb"))

## Paths
Everything is read from the original VOC dataset, so this notebook doesn't depend on any other notebook's outputs.

In [ ]:
VOC_ROOT = Path("/kaggle/input/datasets/gopalbhattrai/pascal-voc-2012-dataset")
TRAIN_DIR = VOC_ROOT / "VOC2012_train_val/VOC2012_train_val"   # Annotations/, JPEGImages/, ImageSets/
TEST_DIR = VOC_ROOT / "VOC2012_test/VOC2012_test"               # Annotations/, JPEGImages/
CKPT_DIR = Path("/kaggle/working/frcnn_checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
PROJECT = "voc-frcnn"   # W&B project

NAMES = ("aeroplane,bicycle,bird,boat,bottle,bus,car,cat,chair,cow,diningtable,"
         "dog,horse,motorbike,person,pottedplant,sheep,sofa,train,tvmonitor").split(",")

## Split
Exactly the same procedure as in the YOLO notebook, so both detectors train, validate and test on **identical images**:
trainval.txt (11,540 detection images) → sorted → shuffled with seed 42 → first 80% train, rest val. Test = every XML of the Kaggle test set.

In [ ]:
ids = sorted((TRAIN_DIR / "ImageSets/Main/trainval.txt").read_text().split())
random.Random(42).shuffle(ids)
cut = int(len(ids) * 0.8)
train_ids, val_ids = ids[:cut], ids[cut:]
test_ids = sorted(p.stem for p in (TEST_DIR / "Annotations").glob("*.xml"))
print(len(train_ids), "train |", len(val_ids), "val |", len(test_ids), "test")   # expect 9232 | 2308 | 5138

## Read the XML annotations
Each XML is parsed when its image is loaded, inside the DataLoader workers (in parallel, no waiting up front). VOC boxes are already corners in pixels (`xmin, ymin, xmax, ymax`), which is the format torchvision wants,
so the only changes are:
- `xmin - 1`, `ymin - 1`: VOC counts pixels from 1, image coordinates start at 0
- class name → index in `NAMES` **+ 1**, because torchvision reserves 0 for background

In [ ]:
def parse_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    boxes, labels = [], []
    for obj in root.findall("object"):
        name = obj.findtext("name").strip()
        bb = obj.find("bndbox")
        x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
        x1, y1 = x1 - 1, y1 - 1
        if x2 > x1 and y2 > y1:
            boxes.append([x1, y1, x2, y2])
            labels.append(NAMES.index(name) + 1)
    return boxes, labels

print(TRAIN_DIR.name, "example", train_ids[0], "→", parse_xml(TRAIN_DIR / "Annotations" / f"{train_ids[0]}.xml"))

### Fixed subsets
Same seeds as the YOLO notebook, drawn from the same train list in the same order, so these are the **same images YOLO used**.
- 32 images for the overfit sanity checks (train = val)
- 1,000 train images for the train-vs-val gap
- 10% of train (random) for the timing run

In [ ]:
random.seed(0)
overfit_ids = random.sample(train_ids, 32)

random.seed(1)
train_eval_ids = random.sample(train_ids, 1000)

random.seed(2)
timing_ids = random.sample(train_ids, len(train_ids) // 10)

## Config
**Starting values: your call.** Every run cell passes them explicitly, so changing a value here changes every run that uses it.

In [ ]:
SEED = 0
EPOCHS = 8
BATCH = 8
NUM_WORKERS = 4

LR = 0.01
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

ETA_MIN_FACTOR = 0.01


MIN_SIZE = 800
MAX_SIZE = 1333


HFLIP_PROB = 0.5
COLOR_JITTER = ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.02)

USE_AMP = True
EVAL_SCORE_THRESH = 0.001
SANITY_EPOCHS = 50

## Dataset
Returns one `(image, target)` pair: the image as a float tensor `[3, H, W]` with values 0–1, and the boxes/labels from its XML.

Augmentation (only when `augment=True`): horizontal flip **with the boxes flipped too**, and colour jitter (image only).

In [ ]:
class VOCDetection(Dataset):
    def __init__(self, ids, image_dir, ann_dir, augment=False):
        self.ids = ids
        self.image_dir = image_dir         # JPEGImages/
        self.ann_dir = ann_dir             # Annotations/ (XML)
        self.augment = augment

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        image_id = self.ids[i]
        image = read_image(str(self.image_dir / f"{image_id}.jpg"), mode=ImageReadMode.RGB).float() / 255  # [3, H, W]
        _, H, W = image.shape

        boxes, labels = parse_xml(self.ann_dir / f"{image_id}.xml")
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.tensor(labels, dtype=torch.int64)

        if self.augment:
            if torch.rand(1).item() < HFLIP_PROB:
                image = image.flip(-1)                         # mirror left-right
                boxes[:, [0, 2]] = W - boxes[:, [2, 0]]        # new x1 = W - old x2, new x2 = W - old x1
            image = COLOR_JITTER(image)

        return image, {"boxes": boxes, "labels": labels}


def collate(batch):
    return tuple(zip(*batch))

### DataLoaders

In [ ]:
TRAIN_JPEG, TRAIN_XML = TRAIN_DIR / "JPEGImages", TRAIN_DIR / "Annotations"
TEST_JPEG, TEST_XML = TEST_DIR / "JPEGImages", TEST_DIR / "Annotations"

def loader(ids, image_dir, ann_dir, augment, shuffle):
    return DataLoader(VOCDetection(ids, image_dir, ann_dir, augment), batch_size=BATCH, shuffle=shuffle,
                      num_workers=NUM_WORKERS, collate_fn=collate)

train_loader = loader(train_ids, TRAIN_JPEG, TRAIN_XML, augment=False, shuffle=True)
train_loader_aug = loader(train_ids, TRAIN_JPEG, TRAIN_XML, augment=True, shuffle=True)
val_loader = loader(val_ids, TRAIN_JPEG, TRAIN_XML, augment=False, shuffle=False)
train_eval_loader = loader(train_eval_ids, TRAIN_JPEG, TRAIN_XML, augment=False, shuffle=False)
overfit_loader = loader(overfit_ids, TRAIN_JPEG, TRAIN_XML, augment=False, shuffle=True)
timing_loader = loader(timing_ids, TRAIN_JPEG, TRAIN_XML, augment=False, shuffle=True)
test_loader = loader(test_ids, TEST_JPEG, TEST_XML, augment=False, shuffle=False)

### Data check
One batch through the dataset: shapes, value ranges and labels must look right before any training.

In [ ]:
images, targets = next(iter(val_loader))
print("images in batch:", len(images), "| first image:", tuple(images[0].shape), f"values {images[0].min():.2f}-{images[0].max():.2f}")
print("first target boxes:\n", targets[0]["boxes"])
print("first target labels:", targets[0]["labels"].tolist(), "→", [NAMES[l - 1] for l in targets[0]["labels"].tolist()])

## Helper functions
Model setup, the training loop (like `train_model` in the FER notebook), evaluation and logging.

In [ ]:
def build_model(trainable_backbone_layers):
    """COCO-pretrained Faster R-CNN with a new 21-class box predictor (20 VOC classes + background)."""
    model = fasterrcnn_mobilenet_v3_large_fpn(
        weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.COCO_V1,
        trainable_backbone_layers=trainable_backbone_layers,   # 0 = backbone frozen ... 6 = all backbone layers train
        min_size=MIN_SIZE, max_size=MAX_SIZE,
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features          # 1024
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, len(NAMES) + 1)

    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"trainable parameters: {n_trainable / 1e6:.2f} M of {n_total / 1e6:.2f} M")
    return model.to(DEVICE), n_trainable, n_total

In [ ]:
@torch.no_grad()
def evaluate(model, data_loader, class_metrics=False):
    """COCO-style mAP with torchmetrics. Returns (metrics dict, per-class W&B table or None)."""
    model.eval()
    model.roi_heads.score_thresh = EVAL_SCORE_THRESH
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox", class_metrics=class_metrics)

    for images, targets in data_loader:
        images = [img.to(DEVICE) for img in images]
        with torch.autocast(DEVICE.type, enabled=USE_AMP):
            preds = model(images)
        preds = [{"boxes": p["boxes"].float().cpu(), "scores": p["scores"].float().cpu(), "labels": p["labels"].cpu()}
                 for p in preds]
        metric.update(preds, list(targets))

    r = metric.compute()
    metrics = {"mAP50-95": r["map"].item(), "mAP50": r["map_50"].item(), "mAP75": r["map_75"].item(),
               "mAP_small": r["map_small"].item(), "mAP_medium": r["map_medium"].item(),
               "mAP_large": r["map_large"].item(), "mAR100": r["mar_100"].item()}
    table = None
    if class_metrics:
        rows = [[NAMES[c - 1], ap] for c, ap in zip(r["classes"].tolist(), r["map_per_class"].tolist())]
        table = wandb.Table(columns=["class", "AP50-95"], data=rows)
    return metrics, table

In [ ]:
def train_model(model, optimizer, scheduler, train_loader, val_loader, epochs, ckpt_path):
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)
    best_map, history = -1.0, []

    for epoch in range(1, epochs + 1):
        model.train()
        t0 = time.time()
        loss_sums, n_batches = {}, 0
        for images, targets in train_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            with torch.autocast(DEVICE.type, enabled=USE_AMP):
                loss_dict = model(images, targets)          # 4 losses: RPN (2) + RoI head (2)
                loss = sum(loss_dict.values())
            if not math.isfinite(loss.item()):
                raise RuntimeError(f"loss is {loss.item()} at epoch {epoch}: lower the learning rate")

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            for k, v in loss_dict.items():
                loss_sums[k] = loss_sums.get(k, 0.0) + v.item()
            loss_sums["loss_total"] = loss_sums.get("loss_total", 0.0) + loss.item()
            n_batches += 1

        lr = optimizer.param_groups[0]["lr"]
        if scheduler is not None:
            scheduler.step()                                 # one scheduler step per epoch
        train_sec = time.time() - t0

        t1 = time.time()
        val, _ = evaluate(model, val_loader)
        val_sec = time.time() - t1

        log = {"epoch": epoch, "lr": lr, "time/train_sec": train_sec, "time/val_sec": val_sec,
               **{f"train/{k}": v / n_batches for k, v in loss_sums.items()},
               **{f"val/{k}": v for k, v in val.items()}}
        wandb.log(log)
        history.append(log)

        if val["mAP50-95"] > best_map:
            best_map = val["mAP50-95"]
            torch.save(model.state_dict(), ckpt_path)

        print(f"epoch {epoch}/{epochs} | loss {log['train/loss_total']:.3f} | val mAP50-95 {val['mAP50-95']:.3f} "
              f"mAP50 {val['mAP50']:.3f} | train {train_sec / 60:.1f} min, val {val_sec / 60:.1f} min")
    return history

In [ ]:
def log_fit_summary(history, epochs):
    maps = [h["val/mAP50-95"] for h in history]
    best = max(range(len(maps)), key=lambda i: maps[i])
    summary = {
        "fit/epochs_run": len(history),
        "fit/best_epoch": best + 1,
        "fit/peak_val_mAP50-95": maps[best],
        "fit/last_val_mAP50-95": maps[-1],
        "fit/drop_from_peak": maps[best] - maps[-1],     # > 0: val got worse at the end -> overfitting hint
        "time/mean_train_sec_per_epoch": sum(h["time/train_sec"] for h in history) / len(history),
        "time/mean_val_sec_per_epoch": sum(h["time/val_sec"] for h in history) / len(history),
    }
    wandb.run.summary.update(summary)
    print(f"best epoch {best + 1} | peak val mAP50-95 {maps[best]:.3f} | drop from peak {summary['fit/drop_from_peak']:.3f}")

In [ ]:
def log_val_and_gap(model, ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    val_m, val_table = evaluate(model, val_loader, class_metrics=True)
    train_m, _ = evaluate(model, train_eval_loader)
    gap = train_m["mAP50-95"] - val_m["mAP50-95"]

    wandb.run.summary.update({f"final/val_{k}": v for k, v in val_m.items()})
    wandb.run.summary.update({f"final/train_{k}": v for k, v in train_m.items()})
    wandb.run.summary["final/gap_mAP50-95"] = gap
    wandb.log({"per_class/val": val_table})
    print(f"val mAP50-95 {val_m['mAP50-95']:.3f} | train mAP50-95 {train_m['mAP50-95']:.3f} | gap {gap:+.3f}")

## Test run — time one epoch
1 epoch on 10% of train (random), then a full validation. Gives minutes per full epoch to set `EPOCHS`, and checks logging.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(trainable_backbone_layers=3)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

wandb.init(project=PROJECT, name="Pilot | FRCNN MobileNetV3 | 10% data, 1 epoch", group="pilot",
           config={"purpose": "time one epoch + test logging", "fraction": 0.1, "batch": BATCH, "min_size": MIN_SIZE})
history = train_model(model, optimizer, None, timing_loader, val_loader, epochs=1, ckpt_path=CKPT_DIR / "pilot.pt")

full_epoch_sec = history[0]["time/train_sec"] / 0.1 + history[0]["time/val_sec"]
wandb.run.summary["pilot/est_full_epoch_min"] = full_epoch_sec / 60
print(f"Estimated full epoch: {full_epoch_sec / 60:.1f} min | 8 epochs ≈ {8 * full_epoch_sec / 3600:.1f} h")
wandb.finish()

### Sanity check — baseline (default: first 3 of 6 backbone stages frozen)
Train and validate on the same 32 images, no augmentation, constant LR, `SANITY_EPOCHS` epochs.
PASS if peak mAP50 ≥ 0.9: this setup can learn at all.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(trainable_backbone_layers=3)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

wandb.init(project=PROJECT, name="Sanity | overfit 32 imgs | FRCNN baseline", group="sanity", config={
    "purpose": "overfit 32 images", "trainable_backbone_layers": 3, "trainable_params": n_trainable,
    "optimizer": "SGD", "lr": LR, "epochs": SANITY_EPOCHS,
})
history = train_model(model, optimizer, None, overfit_loader, overfit_loader, SANITY_EPOCHS, CKPT_DIR / "sanity_baseline.pt")

peak = max(h["val/mAP50"] for h in history)
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

## Run 1 — Baseline
torchvision defaults: last 3 of 6 backbone stages train (`trainable_backbone_layers=3`), new 21-class predictor, SGD with constant LR, **no augmentation, no scheduler**.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(trainable_backbone_layers=3)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = None                 # constant learning rate

wandb.init(project=PROJECT, name="R1 | FRCNN MobileNetV3 baseline", group="ablation", config={
    "run": "R1",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": "new 21-class box predictor (random init)",
    "trainable_backbone_layers": 3, "frozen": "backbone blocks 0-6 (first 3 of 6 stages)",
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "none (constant LR)", "eta_min_factor": ETA_MIN_FACTOR if False else None,
    "augmentation": "none",
    "epochs": EPOCHS, "batch": BATCH, "min_size": MIN_SIZE, "max_size": MAX_SIZE, "amp": USE_AMP,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader, val_loader, EPOCHS, CKPT_DIR / "r1_baseline.pt")

log_fit_summary(history, EPOCHS)
log_val_and_gap(model, CKPT_DIR / "r1_baseline.pt")
wandb.finish()

## Run 2 — Baseline + augmentation
Run 1 + horizontal flip (boxes flipped too) and colour jitter. Only the training images change.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(trainable_backbone_layers=3)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = None                 # constant learning rate

wandb.init(project=PROJECT, name="R2 | FRCNN + augmentation", group="ablation", config={
    "run": "R2",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": "new 21-class box predictor (random init)",
    "trainable_backbone_layers": 3, "frozen": "backbone blocks 0-6 (first 3 of 6 stages)",
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "none (constant LR)", "eta_min_factor": ETA_MIN_FACTOR if False else None,
    "augmentation": "horizontal flip (p=HFLIP_PROB) + colour jitter",
    "epochs": EPOCHS, "batch": BATCH, "min_size": MIN_SIZE, "max_size": MAX_SIZE, "amp": USE_AMP,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader_aug, val_loader, EPOCHS, CKPT_DIR / "r2_aug.pt")

log_fit_summary(history, EPOCHS)
log_val_and_gap(model, CKPT_DIR / "r2_aug.pt")
wandb.finish()

## Run 3 — Baseline + augmentation + scheduler
Run 2 + cosine learning-rate schedule: LR falls from `LR` to `LR × ETA_MIN_FACTOR` over the run, one step per epoch.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(trainable_backbone_layers=3)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * ETA_MIN_FACTOR)

wandb.init(project=PROJECT, name="R3 | FRCNN + augmentation + scheduler", group="ablation", config={
    "run": "R3",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": "new 21-class box predictor (random init)",
    "trainable_backbone_layers": 3, "frozen": "backbone blocks 0-6 (first 3 of 6 stages)",
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "cosine, per epoch, LR -> LR * ETA_MIN_FACTOR", "eta_min_factor": ETA_MIN_FACTOR if True else None,
    "augmentation": "horizontal flip (p=HFLIP_PROB) + colour jitter",
    "epochs": EPOCHS, "batch": BATCH, "min_size": MIN_SIZE, "max_size": MAX_SIZE, "amp": USE_AMP,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader_aug, val_loader, EPOCHS, CKPT_DIR / "r3_aug_sched.pt")

log_fit_summary(history, EPOCHS)
log_val_and_gap(model, CKPT_DIR / "r3_aug_sched.pt")
wandb.finish()

### Sanity check — backbone fully frozen
Train and validate on the same 32 images, no augmentation, constant LR, `SANITY_EPOCHS` epochs.
PASS if peak mAP50 ≥ 0.9: this setup can learn at all.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(trainable_backbone_layers=0)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

wandb.init(project=PROJECT, name="Sanity | overfit 32 imgs | FRCNN frozen backbone", group="sanity", config={
    "purpose": "overfit 32 images", "trainable_backbone_layers": 0, "trainable_params": n_trainable,
    "optimizer": "SGD", "lr": LR, "epochs": SANITY_EPOCHS,
})
history = train_model(model, optimizer, None, overfit_loader, overfit_loader, SANITY_EPOCHS, CKPT_DIR / "sanity_frozen_backbone.pt")

peak = max(h["val/mAP50"] for h in history)
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

## Run 4 — Run 3 with the backbone fully frozen
`trainable_backbone_layers=0`: no MobileNetV3 layer updates. FPN, RPN and RoI head still train.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(trainable_backbone_layers=0)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * ETA_MIN_FACTOR)

wandb.init(project=PROJECT, name="R4 | FRCNN aug + sched + frozen backbone", group="ablation", config={
    "run": "R4",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": "new 21-class box predictor (random init)",
    "trainable_backbone_layers": 0, "frozen": "whole backbone",
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "cosine, per epoch, LR -> LR * ETA_MIN_FACTOR", "eta_min_factor": ETA_MIN_FACTOR if True else None,
    "augmentation": "horizontal flip (p=HFLIP_PROB) + colour jitter",
    "epochs": EPOCHS, "batch": BATCH, "min_size": MIN_SIZE, "max_size": MAX_SIZE, "amp": USE_AMP,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader_aug, val_loader, EPOCHS, CKPT_DIR / "r4_frozen_backbone.pt")

log_fit_summary(history, EPOCHS)
log_val_and_gap(model, CKPT_DIR / "r4_frozen_backbone.pt")
wandb.finish()

### Sanity check — whole backbone trainable
Train and validate on the same 32 images, no augmentation, constant LR, `SANITY_EPOCHS` epochs.
PASS if peak mAP50 ≥ 0.9: this setup can learn at all.

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(trainable_backbone_layers=6)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

wandb.init(project=PROJECT, name="Sanity | overfit 32 imgs | FRCNN full backbone", group="sanity", config={
    "purpose": "overfit 32 images", "trainable_backbone_layers": 6, "trainable_params": n_trainable,
    "optimizer": "SGD", "lr": LR, "epochs": SANITY_EPOCHS,
})
history = train_model(model, optimizer, None, overfit_loader, overfit_loader, SANITY_EPOCHS, CKPT_DIR / "sanity_full_backbone.pt")

peak = max(h["val/mAP50"] for h in history)
wandb.run.summary["sanity/peak_mAP50"] = peak
print("PASS" if peak >= 0.9 else "FAIL", f"— peak mAP50 on the memorized images: {peak:.3f}")
wandb.finish()

## Run 5 — Run 3 with the whole backbone trainable
`trainable_backbone_layers=6`: every MobileNetV3 stage updates (BatchNorm statistics stay frozen).

In [ ]:
torch.manual_seed(SEED)
model, n_trainable, n_total = build_model(trainable_backbone_layers=6)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR * ETA_MIN_FACTOR)

wandb.init(project=PROJECT, name="R5 | FRCNN aug + sched + full backbone", group="ablation", config={
    "run": "R5",
    "architecture": "Faster R-CNN MobileNetV3-Large-FPN", "pretrained": "COCO",
    "head": "new 21-class box predictor (random init)",
    "trainable_backbone_layers": 6, "frozen": "nothing (BatchNorm stats only)",
    "trainable_params": n_trainable, "total_params": n_total,
    "optimizer": "SGD", "lr": LR, "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
    "scheduler": "cosine, per epoch, LR -> LR * ETA_MIN_FACTOR", "eta_min_factor": ETA_MIN_FACTOR if True else None,
    "augmentation": "horizontal flip (p=HFLIP_PROB) + colour jitter",
    "epochs": EPOCHS, "batch": BATCH, "min_size": MIN_SIZE, "max_size": MAX_SIZE, "amp": USE_AMP,
    "eval_score_thresh": EVAL_SCORE_THRESH, "seed": SEED,
    "split": "VOC2012 trainval 80/20, seed 42", "map_implementation": "torchmetrics (COCO-style)",
})
history = train_model(model, optimizer, scheduler, train_loader_aug, val_loader, EPOCHS, CKPT_DIR / "r5_full_backbone.pt")

log_fit_summary(history, EPOCHS)
log_val_and_gap(model, CKPT_DIR / "r5_full_backbone.pt")
wandb.finish()

## Final test evaluation — run ONCE, at the end
Pick the run to report. Set its checkpoint, the `trainable_backbone_layers` it used, and its W&B run ID (from the run URL).

In [ ]:
RUN_ID = "PASTE_RUN_ID"
CKPT = CKPT_DIR / "r3_aug_sched.pt"
TRAINABLE_BACKBONE_LAYERS = 3

model, _, _ = build_model(trainable_backbone_layers=TRAINABLE_BACKBONE_LAYERS)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
metrics, table = evaluate(model, test_loader, class_metrics=True)

wandb.init(project=PROJECT, id=RUN_ID, resume="must")   # add test results to the same W&B run
wandb.run.summary.update({f"test/{k}": v for k, v in metrics.items()})
wandb.log({"per_class/test": table})
wandb.finish()
print(metrics)